In [1]:
!pip install torch pandas numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 82.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 62.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 76.6 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子以保证结果可复现
torch.manual_seed(42)
np.random.seed(42)

In [3]:
# 因为是中文所以要用gbk打开
train_df = pd.read_csv('/kaggle/input/traintestdataset/train.csv', encoding='gbk')
test_df = pd.read_csv('/kaggle/input/traintestdataset/test.csv', encoding='gbk')

In [4]:
train_df.head()

,age,browse_duration,purchase_count,avg_order_value,gender,city_tier,device_type,return_rate,coupon_usage,favorite_count,click_through_rate,last_purchase_days,app_install_count,membership_level,social_media_active,has_children,education_level,income_bracket,label
0,28.0,300.0,43.0,1799.0,女,4.0,iOS,0.01,10.0,60.0,0.87,327.0,0.0,普通,否,是,NaN,高,1
1,28.0,225.0,47.0,1461.0,男,4.0,Web,0.08,16.0,18.0,0.07,129.0,1.0,普通,是,否,硕士,低,1
2,41.0,236.0,22.0,321.0,女,2.0,NaN,0.02,10.0,3.0,0.46,101.0,4.0,普通,否,是,本科,中,1
3,41.0,198.0,20.0,955.0,男,2.0,Web,0.04,14.0,45.0,0.27,32.0,4.0,普通,否,否,本科,中,1
4,NaN,284.0,40.0,1519.0,女,2.0,Android,NaN,12.0,99.0,0.52,NaN,5.0,普通,是,否,本科,中,1


In [5]:
# 检查数据是否平衡
num_label_1 = (train_df['label'] == 1).sum()
print("Number of rows with label = 1:", num_label_1)
num_label_0 = (train_df['label'] == 0).sum()
print("Number of rows with label = 0:", num_label_0)

Number of rows with label = 1: 2398
Number of rows with label = 0: 5602


可以看出数据是不平衡的，所以我们需要裁撤将近一半label=0的数据

In [6]:
# 因为 2398 * 2 = 4796 所以我们取前4800个数据
new_train_df = train_df.iloc[:4800]
validate_df = train_df.iloc[4800:]
new_train_df.head()

,age,browse_duration,purchase_count,avg_order_value,gender,city_tier,device_type,return_rate,coupon_usage,favorite_count,click_through_rate,last_purchase_days,app_install_count,membership_level,social_media_active,has_children,education_level,income_bracket,label
0,28.0,300.0,43.0,1799.0,女,4.0,iOS,0.01,10.0,60.0,0.87,327.0,0.0,普通,否,是,NaN,高,1
1,28.0,225.0,47.0,1461.0,男,4.0,Web,0.08,16.0,18.0,0.07,129.0,1.0,普通,是,否,硕士,低,1
2,41.0,236.0,22.0,321.0,女,2.0,NaN,0.02,10.0,3.0,0.46,101.0,4.0,普通,否,是,本科,中,1
3,41.0,198.0,20.0,955.0,男,2.0,Web,0.04,14.0,45.0,0.27,32.0,4.0,普通,否,否,本科,中,1
4,NaN,284.0,40.0,1519.0,女,2.0,Android,NaN,12.0,99.0,0.52,NaN,5.0,普通,是,否,本科,中,1


In [7]:
# 定义特征类型
numeric_features = ['age', 'browse_duration', 'purchase_count', 'avg_order_value', 
                    'return_rate', 'coupon_usage', 'favorite_count', 'click_through_rate',
                    'last_purchase_days', 'app_install_count']
categorical_features = ['gender', 'city_tier', 'device_type', 'membership_level',
                        'social_media_active', 'has_children', 'education_level', 'income_bracket']

In [8]:
# 处理缺失值
# 数值特征：用中位数填充
num_imputer = SimpleImputer(strategy='median')
new_train_df[numeric_features] = num_imputer.fit_transform(new_train_df[numeric_features])

# 类别特征：用最频繁值填充
cat_imputer = SimpleImputer(strategy='most_frequent')
new_train_df[categorical_features] = cat_imputer.fit_transform(new_train_df[categorical_features])

In [9]:
# 标准化数值特征
scaler = StandardScaler()
new_train_df[numeric_features] = scaler.fit_transform(new_train_df[numeric_features])


In [10]:
# 编码类别特征
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    new_train_df[col] = le.fit_transform(new_train_df[col].astype(str))
    label_encoders[col] = le

In [11]:
# 创建增强型数据集
class EnhancedCustomerDataset(Dataset):
    def __init__(self, data, labels, numeric_cols, categorical_cols):
        self.numeric_data = data[numeric_cols].values.astype(np.float32)
        self.categorical_data = data[categorical_cols].values.astype(np.int64)
        self.labels = labels.values.astype(np.float32)
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return {
            'numeric': self.numeric_data[idx],
            'categorical': self.categorical_data[idx],
            'label': self.labels[idx]
        }

In [12]:
#定义增强型 Wide & Deep 模型
class FeatureEmbedding(nn.Module):
    """为每个类别特征创建嵌入层"""
    def __init__(self, categorical_dims, embedding_dim):
        super(FeatureEmbedding, self).__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(dim, embedding_dim) for dim in categorical_dims
        ])
        
    def forward(self, x_categorical):
        embeddings = []
        for i, emb in enumerate(self.embeddings):
            embeddings.append(emb(x_categorical[:, i]))
        return torch.cat(embeddings, dim=1)

class FeatureAttention(nn.Module):
    """特征级自注意力机制"""
    def __init__(self, feature_dim, num_heads=1, dropout=0.1):
        super(FeatureAttention, self).__init__()
        print(feature_dim)
        self.attention = nn.MultiheadAttention(
            embed_dim=feature_dim, 
            num_heads=num_heads, 
            dropout=dropout,
            batch_first=True
        )
        self.norm = nn.LayerNorm(feature_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # 添加序列维度 (batch_size, seq_len=1, feature_dim)
        x = x.unsqueeze(1)
        attn_output, _ = self.attention(x, x, x)
        attn_output = self.dropout(attn_output)
        x = self.norm(x + attn_output)
        return x.squeeze(1)


In [13]:
class EnhancedWideAndDeep(nn.Module):
    def __init__(self, numeric_dim, categorical_dims, embedding_dim=8, 
                 wide_dim=None, hidden_dims=[64, 32], num_attention_heads=2, dropout_rate=0.2):
        super(EnhancedWideAndDeep, self).__init__()
        
        # 宽部分 (处理类别特征)
        self.wide_dim = wide_dim if wide_dim else len(categorical_dims)
        self.wide = nn.Linear(self.wide_dim, 1)
        
        # 深部分 - 特征嵌入
        self.embedding = FeatureEmbedding(categorical_dims, embedding_dim)
        
        # 计算深部分输入维度
        self.deep_input_dim = numeric_dim + len(categorical_dims) * embedding_dim
        
        # 特征注意力机制
        self.feature_attention = FeatureAttention(self.deep_input_dim, num_attention_heads, dropout_rate)
        
        # 深部分MLP
        deep_layers = []
        input_dim = self.deep_input_dim
        
        for hidden_dim in hidden_dims:
            deep_layers.append(nn.Linear(input_dim, hidden_dim))
            deep_layers.append(nn.BatchNorm1d(hidden_dim))
            deep_layers.append(nn.ReLU())
            deep_layers.append(nn.Dropout(dropout_rate))
            input_dim = hidden_dim
            
        self.deep_mlp = nn.Sequential(*deep_layers)
        
        # 交互层 - 学习宽部分和深部分之间的交互
        self.interaction_layer = nn.Linear(1 + hidden_dims[-1], 16)
        self.interaction_activation = nn.ReLU()
        
        # 输出层
        self.output = nn.Linear(16, 1)
        
    def forward(self, numeric_x, categorical_x):
        # 宽部分
        wide_output = self.wide(categorical_x.float())
        
        # 深部分
        embedded = self.embedding(categorical_x)
        deep_input = torch.cat([numeric_x, embedded], dim=1)
        
        # 应用特征注意力
        deep_attended = self.feature_attention(deep_input)
        
        # 通过MLP
        deep_output = self.deep_mlp(deep_attended)
        
        # 合并宽和深部分
        combined = torch.cat([wide_output, deep_output], dim=1)
        
        # 交互层
        interaction = self.interaction_activation(self.interaction_layer(combined))
        
        # 输出
        output = torch.sigmoid(self.output(interaction))
        
        return output.squeeze()

In [14]:
# 训练和评估函数
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler=None, num_epochs=50):
    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    patience_counter = 0
    patience = 10  # 早停耐心值
    
    for epoch in range(num_epochs):
        # 训练阶段
        model.train()
        running_loss = 0.0
        
        for batch in train_loader:
            numeric_x = batch['numeric']
            categorical_x = batch['categorical']
            labels = batch['label']
            
            optimizer.zero_grad()
            outputs = model(numeric_x, categorical_x)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # 梯度裁剪防止爆炸
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            running_loss += loss.item() * numeric_x.size(0)
        
        epoch_loss = running_loss / len(train_loader.dataset)
        train_losses.append(epoch_loss)
        
        # 验证阶段
        model.eval()
        val_loss = 0.0
        all_preds, all_labels = [], []
        
        with torch.no_grad():
            for batch in val_loader:
                numeric_x = batch['numeric']
                categorical_x = batch['categorical']
                labels = batch['label']
                
                outputs = model(numeric_x, categorical_x)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * numeric_x.size(0)
                
                preds = (outputs > 0.5).float()
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        val_epoch_loss = val_loss / len(val_loader.dataset)
        val_losses.append(val_epoch_loss)
        
        val_accuracy = accuracy_score(all_labels, all_preds)
        
        # 学习率调度
        if scheduler:
            scheduler.step(val_epoch_loss)
        
        # 早停检查
        if val_epoch_loss < best_val_loss:
            best_val_loss = val_epoch_loss
            patience_counter = 0
            # 保存最佳模型
            torch.save(model.state_dict(), 'best_model.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"早停触发于第 {epoch+1} 轮")
                break
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], '
                  f'Train Loss: {epoch_loss:.4f}, Val Loss: {val_epoch_loss:.4f}, '
                  f'Val Accuracy: {val_accuracy:.4f}')
    
    return train_losses, val_losses

In [15]:
def evaluate_model(model, test_loader):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    
    with torch.no_grad():
        for batch in test_loader:
            numeric_x = batch['numeric']
            categorical_x = batch['categorical']
            labels = batch['label']
            
            outputs = model(numeric_x, categorical_x)
            probs = outputs.cpu().numpy()
            preds = (outputs > 0.5).float().cpu().numpy()
            
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    
    accuracy = accuracy_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)
    report = classification_report(all_labels, all_preds)
    cm = confusion_matrix(all_labels, all_preds)
    
    print(f"测试集准确率: {accuracy:.4f}")
    print(f"测试集AUC: {auc:.4f}")
    print("\n分类报告:")
    print(report)
    
    # 绘制混淆矩阵
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('confusion matrix')
    plt.ylabel('真实标签')
    plt.xlabel('预测标签')
    plt.savefig('confusion_matrix.png')
    plt.close()
    
    return all_probs, all_preds, all_labels

In [16]:
#交叉验证训练
def cross_validate(model_class, X, y, numeric_cols, categorical_cols, 
                   categorical_dims, n_splits=5, batch_size=32, num_epochs=50):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_results = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n Training {fold+1}/{n_splits} folds...")
        
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        # 创建数据加载器
        train_dataset = EnhancedCustomerDataset(X_train, y_train, numeric_cols, categorical_cols)
        val_dataset = EnhancedCustomerDataset(X_val, y_val, numeric_cols, categorical_cols)
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        print(len(numeric_cols))
        # 初始化模型
        model = model_class(
            numeric_dim=len(numeric_cols),
            categorical_dims=categorical_dims,
            embedding_dim=8,
            hidden_dims=[64, 32],
            num_attention_heads=2,
            dropout_rate=0.3
        )
        
        criterion = nn.BCELoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
        
        # 训练模型
        train_losses, val_losses = train_model(
            model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs
        )
        
        # 评估模型
        probs, preds, labels = evaluate_model(model, val_loader)
        fold_results.append({
            'accuracy': accuracy_score(labels, preds),
            'auc': roc_auc_score(labels, probs),
            'train_losses': train_losses,
            'val_losses': val_losses
        })
    
    return fold_results


In [17]:
if __name__ == "__main__":
    # 准备数据
    X = new_train_df.drop('label', axis=1)
    y = new_train_df['label']
    
    # 获取类别特征的维度
    categorical_dims = [len(label_encoders[col].classes_) for col in categorical_features]
    print("开始交叉验证训练...")
    results = cross_validate(
        EnhancedWideAndDeep, X, y, numeric_features, categorical_features, 
        categorical_dims, n_splits=5, batch_size=32, num_epochs=50
    )
    
    # 计算平均性能
    avg_accuracy = np.mean([r['accuracy'] for r in results])
    avg_auc = np.mean([r['auc'] for r in results])
    
    print(f"\n交叉验证平均准确率: {avg_accuracy:.4f}")
    print(f"交叉验证平均AUC: {avg_auc:.4f}")
    
    # 绘制训练曲线 (使用最后一折的结果)
    plt.figure(figsize=(10, 6))
    plt.plot(results[-1]['train_losses'], label='训练损失')
    plt.plot(results[-1]['val_losses'], label='验证损失')
    plt.title('训练和验证损失曲线')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig('training_curve.png')
    plt.close()
    
    # 加载最佳模型进行最终测试
    print("\n使用最佳模型进行测试...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    train_dataset = EnhancedCustomerDataset(X_train, y_train, numeric_features, categorical_features)
    test_dataset = EnhancedCustomerDataset(X_test, y_test, numeric_features, categorical_features)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    
    # 初始化并加载最佳模型
    best_model = EnhancedWideAndDeep(
        numeric_dim=len(numeric_features),
        categorical_dims=categorical_dims,
        embedding_dim=8,
        hidden_dims=[64, 32],
        num_attention_heads=2,
        dropout_rate=0.3
    )
    best_model.load_state_dict(torch.load('best_model.pth'))
    
    # 最终评估
    test_probs, test_preds, test_labels = evaluate_model(best_model, test_loader)
    
    # 绘制概率分布
    plt.figure(figsize=(10, 6))
    plt.hist([test_probs[i] for i in range(len(test_probs)) if test_labels[i] == 0], 
             alpha=0.5, label='负例', bins=20)
    plt.hist([test_probs[i] for i in range(len(test_probs)) if test_labels[i] == 1], 
             alpha=0.5, label='正例', bins=20)
    plt.title('预测概率分布')
    plt.xlabel('预测概率')
    plt.ylabel('频数')
    plt.legend()
    plt.savefig('probability_distribution.png')
    plt.close()

开始交叉验证训练...

 Training 1/5 folds...
10
74
Epoch [10/50], Train Loss: 0.3569, Val Loss: 0.3039, Val Accuracy: 0.8719
Epoch [20/50], Train Loss: 0.3048, Val Loss: 0.2875, Val Accuracy: 0.8927
Epoch [30/50], Train Loss: 0.2915, Val Loss: 0.2950, Val Accuracy: 0.8854
早停触发于第 33 轮
测试集准确率: 0.8823
测试集AUC: 0.9507

分类报告:
              precision    recall  f1-score   support

         0.0       0.96      0.80      0.87       480
         1.0       0.83      0.97      0.89       480

    accuracy                           0.88       960
   macro avg       0.89      0.88      0.88       960
weighted avg       0.89      0.88      0.88       960


 Training 2/5 folds...
10
74
Epoch [10/50], Train Loss: 0.3571, Val Loss: 0.3350, Val Accuracy: 0.8500
Epoch [20/50], Train Loss: 0.3087, Val Loss: 0.3074, Val Accuracy: 0.8677
Epoch [30/50], Train Loss: 0.2969, Val Loss: 0.2722, Val Accuracy: 0.8948
早停触发于第 38 轮
测试集准确率: 0.8958
测试集AUC: 0.9587

分类报告:
              precision    recall  f1-score   support

    

In [18]:
# 在剩余的训练集上打标签
validate_df[numeric_features] = num_imputer.fit_transform(validate_df[numeric_features])
validate_df[categorical_features] = cat_imputer.fit_transform(validate_df[categorical_features])
validate_df[numeric_features] = scaler.fit_transform(validate_df[numeric_features])
for col in categorical_features:
    le = LabelEncoder()
    validate_df[col] = le.fit_transform(validate_df[col].astype(str))
    label_encoders[col] = le

validate_x = validate_df.drop('label', axis=1)
validate_y = validate_df['label']

def predict_model(model, test_loader, threshold=0.5):
    """
    用训练好的模型对测试集进行预测
    返回：
        all_preds: 二分类预测结果 (0/1)
        all_probs: 概率预测结果 (0~1)
    """
    model.eval()  # 切换到评估模式
    all_probs, all_preds = [], []

    with torch.no_grad():  # 禁用梯度计算
        for batch in test_loader:
            numeric_x = batch['numeric']
            categorical_x = batch['categorical']
            
            # 模型输出
            outputs = model(numeric_x, categorical_x)
            probs = outputs.cpu().numpy()
            preds = (outputs > threshold).float().cpu().numpy()
            
            all_probs.extend(probs)
            all_preds.extend(preds)
    
    return all_preds, all_probs


validate_test_dataset = EnhancedCustomerDataset(validate_x, validate_y, numeric_features, categorical_features)
    
validate_test_loader = DataLoader(validate_test_dataset, batch_size=64, shuffle=False)

validate_test_pred, validate_probs= predict_model(best_model, validate_test_loader)

# 打印出结果
print(validate_test_pred)
print(validate_probs)

[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0,

In [19]:
validate_y_np = validate_y.to_numpy()
validate_test_pred_np = np.array(validate_test_pred)

count = 0
for i in range(len(validate_y_np)):
    if validate_y_np[i] != validate_test_pred_np[i]:
        count += 1

# 计算 accuracy
print(count)
print(1-(count/len(validate_y_np)))

1516
0.52625


In [20]:
# 给test.csv开始打标签
test_df[numeric_features] = num_imputer.fit_transform(test_df[numeric_features])
test_df[categorical_features] = cat_imputer.fit_transform(test_df[categorical_features])
test_df[numeric_features] = scaler.fit_transform(test_df[numeric_features])
for col in categorical_features:
    le = LabelEncoder()
    test_df[col] = le.fit_transform(test_df[col].astype(str))
    label_encoders[col] = le

test_df['label'] = [None] * len(test_df)
test_x = test_df.drop('label', axis=1)
test_y = test_df['label']

test_dataset = EnhancedCustomerDataset(test_x, test_y, numeric_features, categorical_features)
    
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

test_pred, test_probs= predict_model(best_model, test_loader)

In [21]:
print(type(test_pred))

<class 'list'>


In [22]:
for pred in test_pred:
    pred = int(pred)

In [23]:
test_label_df = pd.DataFrame({"label": test_pred})
test_label_df

,label
0,0.0
1,1.0
2,0.0
3,0.0
4,0.0
...,...
1995,0.0
1996,1.0
1997,1.0
1998,0.0


In [24]:
# 把标签赋值到label那一列上
print(len(test_df), len(test_pred))
test_df['label'] = test_pred
test_df.head()

2000 2000


,age,browse_duration,purchase_count,avg_order_value,gender,city_tier,device_type,return_rate,coupon_usage,favorite_count,click_through_rate,last_purchase_days,app_install_count,membership_level,social_media_active,has_children,education_level,income_bracket,label
0,0.509917,-1.054017,1.323610,0.366648,1,4,0,1.567094,0.347596,0.674358,0.941400,-1.011040,0.321456,1,0,1,3,0,0.0
1,0.232430,0.151939,-0.343633,1.005658,1,1,0,-0.694551,0.521699,-0.891933,0.475378,0.002725,-0.871330,1,1,0,1,1,1.0
2,0.440545,-0.011978,1.686055,-0.295828,1,2,0,1.416318,-0.697020,-0.717901,-0.313274,0.444366,0.917849,1,1,1,1,1,0.0
3,0.718032,0.912979,-0.851056,-1.534136,0,2,0,-0.091446,-1.045226,-1.657676,1.479118,1.297534,0.917849,2,0,1,1,1,0.0
4,-0.461287,0.315855,-1.720922,-1.395142,0,0,1,-0.166834,0.695802,-0.230610,-0.384970,0.002725,0.321456,1,0,0,3,0,0.0


In [25]:
test_df.to_csv("test_with_pred.csv", index=False, encoding="utf-8")